# Constraint records: atomic vs compound

`fem.nodes.constraints` and `fem.elements.constraints` hold a **mixed**
collection of constraint records. Some of them are **atomic** — one record
equals one solver command. Others are **compound** — one record wraps
several atomic records *plus* side-band data (phantom coordinates,
rigid-arm offsets, mortar operators) that the atomic form cannot carry.

The iterators on these sets come in two flavours:

* **Atomic-returning** iterators (e.g. `pairs()`, `equal_dofs()`,
  `interpolations()`) automatically walk into compound records and yield
  their atomic pieces.
* **Compound-returning** iterators (e.g. `node_to_surfaces()`,
  `couplings()`) yield the outer wrapper so you can reach the side-band
  data.

This notebook builds a minimal model with one compound constraint
(`node_to_surface`), then walks through every iterator to show
**exactly** what each one yields and when you want it.

By the end you should be able to look at a call like
`fem.nodes.constraints.couplings()` and know, without reading the source,
whether it expands or not.

In [ ]:
from apeGmsh import apeGmsh
from apeGmsh.solvers.Constraints import (
    NodePairRecord, NodeGroupRecord, NodeToSurfaceRecord,
)

## 1. Build a minimal model

We want the smallest possible mesh that produces both tiers of records.
A short W-section column with two reference points (at the base and top)
and two `node_to_surface` constraints gives us:

* two **compound** `NodeToSurfaceRecord`s (one per ref point)
* the atomic pairs *inside* each compound record
* phantom nodes as side-band data

Nothing else — no loads, no analysis. This is a data-model tour.

In [ ]:
m = apeGmsh(model_name='tier_demo', verbose=False)
m.begin()

# Simple W-section column — section helper gives us labelled end faces.
column = m.sections.W_solid(
    bf=150, tf=20, h=300, tw=10, length=2000, label='column')
m.model.selection.select_volumes().to_physical(name='pg_column')

# 6-DOF reference points at base and top.
m.model.geometry.add_point(0, 0, 0,    lc=100, label='base')
m.model.geometry.add_point(0, 0, 2000, lc=100, label='top')

# Two compound constraints — each produces one NodeToSurfaceRecord.
m.constraints.node_to_surface('base', column.labels.start_face)
m.constraints.node_to_surface('top',  column.labels.end_face)

m.mesh.sizing.set_global_size(100)
m.mesh.generation.generate(dim=3)

fem = m.mesh.queries.get_fem_data(remove_orphans=True)
m.end()

nc = fem.nodes.constraints
print(f'{len(nc)} record(s) stored in fem.nodes.constraints')

## 2. What is in the set?

Iterating the set directly gives you the records **exactly as they were
resolved** — no expansion, no filtering. Mixed types.

For this model we expect exactly two `NodeToSurfaceRecord` objects,
because we defined two `node_to_surface` constraints.

In [ ]:
from collections import Counter

types = Counter(type(r).__name__ for r in nc)
print('raw set contents:')
for name, n in types.items():
    print(f'  {n:3d} × {name}')

# Every raw record also has .kind and .name for quick identification.
print()
for r in nc:
    print(f'  kind={r.kind!r:20s} name={r.name!r}')

## 3. Atomic iteration — `.pairs()`

`NodeConstraintSet.pairs()` is the **atomic** iterator. Its job is to
yield every constraint as a flat sequence of `NodePairRecord` objects,
one per solver command. Compound records are walked and expanded for you.

A `NodeToSurfaceRecord` expands into:

* N `rigid_beam` pairs (master → each phantom node), plus
* N `equal_dof` pairs (phantom node → original slave)

where N is the number of slave nodes on the surface. So for our two
compound records, the atomic count is `2 · 2 · N_surface_nodes`.

Notice the jump in count between raw set and `.pairs()`:

In [ ]:
print(f'raw records:            {len(nc)}')
print(f'atomic pairs (.pairs):  {sum(1 for _ in nc.pairs())}')

# Every yield is a NodePairRecord — check and peek at the first few.
print()
for i, p in enumerate(nc.pairs()):
    assert isinstance(p, NodePairRecord)
    print(f'  {i:3d}  kind={p.kind:11s}  master={p.master_node:4d}  '
          f'slave={p.slave_node:4d}  dofs={p.dofs}')
    if i >= 5:
        print('  ...')
        break

`.pairs()` is the generic atomic iterator. There are also more
**specific** atomic iterators that only yield one constraint type
or pre-group pairs by master:

| Iterator | Yields | When |
|----------|--------|------|
| `equal_dofs()` | `NodePairRecord(kind='equal_dof')` only | emitting `ops.equalDOF` |
| `rigid_link_groups()` | `(master, [slaves])` tuples | emitting `ops.rigidLink` |
| `rigid_diaphragms()` | `(master, [slaves])` — diaphragm only | emitting `ops.rigidDiaphragm` |

All of these are atomic-returning: they walk compound records for you.

In [ ]:
print(f'equal_dofs():          {sum(1 for _ in nc.equal_dofs())}')
print(f'rigid_link_groups():   {sum(1 for _ in nc.rigid_link_groups())}  '
      f'(grouped by master)')
print()

for master, slaves in nc.rigid_link_groups():
    print(f'  master {master} → {len(slaves)} slaves: '
          f'{slaves[:4]}{"..." if len(slaves) > 4 else ""}')

## 4. Compound iteration — `.node_to_surfaces()`

Compound iterators hand you the **wrapper**, not the atoms inside. You
use them when you need a field that the atomic form throws away.

For `NodeToSurfaceRecord` the side-band data is:

* `phantom_nodes` — list of generated 6-DOF phantom node tags
* `phantom_coords` — ndarray of phantom coordinates (solvers need these
  to declare the nodes *before* emitting the rigid links)
* `rigid_link_records`, `equal_dof_records` — the raw atoms that `.pairs()`
  would have yielded, but kept together so you can correlate them back
  to this particular coupling

None of these are reachable from `.pairs()`:

In [ ]:
for nts in nc.node_to_surfaces():
    assert isinstance(nts, NodeToSurfaceRecord)
    n = len(nts.slave_nodes)
    print(f'{nts.name or "(anon)":18s}  master={nts.master_node}  '
          f'n_slaves={n}')
    print(f'  phantom_nodes[:3]  = {nts.phantom_nodes[:3]}')
    print(f'  phantom_coords[0]  = {nts.phantom_coords[0]}')
    print(f'  rigid_link pairs   = {len(nts.rigid_link_records)}')
    print(f'  equal_dof pairs    = {len(nts.equal_dof_records)}')
    print()

## 5. Side-band shortcut — `.phantom_nodes()`

Phantom nodes are such a common side-band need that the set exposes a
dedicated iterator for them. It returns a `NodeResult` object that
yields `(node_id, xyz)` pairs — ready for `ops.node(nid, *xyz)`:

In [ ]:
pn = nc.phantom_nodes()
print(f'total phantom nodes: {len(pn)}')
print()

# Pair iteration — the emission-friendly one-liner:
for i, (nid, xyz) in enumerate(pn):
    x, y, z = xyz
    print(f'  ops.node({nid}, {x:.1f}, {y:.1f}, {z:.1f})')
    if i >= 3:
        print('  ...')
        break

# Or reach for the bulk arrays — same NodeResult, two access shapes.
print()
print(f'pn.ids.shape    = {pn.ids.shape}')
print(f'pn.coords.shape = {pn.coords.shape}')

## 6. The gotcha — expansion is invisible at the call site

If you skim this block in a solver adapter, nothing warns you that the
two loops walk *different* amounts of data:

```python
for rec in fem.nodes.constraints:          # 2 yields — one per compound
    ...
for pair in fem.nodes.constraints.pairs(): # dozens of yields — flattened
    ...
```

The difference matters when you reach for `fem.elements.constraints`:
`interpolations()` *expands* `SurfaceCouplingRecord.slave_records`, while
`couplings()` yields only the top-level wrappers. Two very different
counts, same naming pattern.

**The rule**:

1. Iterator name sounds like an atomic type (`pair`, `interpolation`,
   `equal_dof`) → it expands compounds.
2. Iterator name sounds like a compound type (`node_to_surface`,
   `coupling`) → it yields raw wrappers.
3. Direct iteration of the set → raw mix. Never expanded.

Two counts to verify the rule on this model:

In [ ]:
direct_count = sum(1 for _ in nc)                  # raw, no expansion
atomic_count = sum(1 for _ in nc.pairs())          # expanded
compound_count = sum(1 for _ in nc.node_to_surfaces())  # compound only

print(f'  direct iteration:         {direct_count:4d}  (raw, no expansion)')
print(f'  .pairs():                 {atomic_count:4d}  (atomic, expands compounds)')
print(f'  .node_to_surfaces():      {compound_count:4d}  (compound wrappers only)')

# Sanity check: atomic count equals twice the number of slave nodes
# (rigid_link + equal_dof for every slave across every compound).
expected = sum(2 * len(r.slave_nodes) for r in nc.node_to_surfaces())
assert atomic_count == expected, (atomic_count, expected)
print(f'\n  check: 2 * sum(n_slaves) = {expected}  → matches .pairs() count')

## 7. Recap — when to reach for which iterator

| You need… | Use | Why |
|---|---|---|
| Flat list of pair constraints to feed the solver | `.pairs()` | Expands compounds, one yield per solver command |
| Just `equal_dof` pairs | `.equal_dofs()` | Same, filtered to one kind |
| Grouped rigid links for `ops.rigidLink` loops | `.rigid_link_groups()` | Atomic pairs pre-grouped by master |
| Phantom node coordinates (to declare before emission) | `.phantom_nodes()` | `NodeResult`, pair-iterates as `(id, xyz)` |
| Any other side-band field of a compound record | compound iter (`.node_to_surfaces()`, `.couplings()`) | Expansion erases these |
| Mixed inspection / debugging | direct `for rec in nc:` | Raw subclass access, no expansion |

**Mental model:** *atomic iterators walk the tree; compound iterators
stop at the wrapper.* If the method name matches an atomic record type,
it walks. If it matches a compound type, it stops.

The same rule applies on `fem.elements.constraints`:
`interpolations()` walks (yields `InterpolationRecord`),
`couplings()` stops (yields `SurfaceCouplingRecord`).